# Export a crop U-Net's probability maps (holdout and test)

Import this notebook into Kaggle and press **Save & Run All**. None of the cells asks for input. With the default `MODEL = "r34"` no cell needs editing. To export another model, change only `MODEL` in the settings cell and attach that model's training output:

| `MODEL` | Encoder | Save the training notebook as | Checkpoint |
| --- | --- | --- | --- |
| `r34` (default) | `resnet34` | `train-unet-r34-crops-80ep` | `unet_r34_crops.pt` |
| `r50` | `resnet50` | `train-unet-r50-crops` | `unet_r50_crops.pt` |
| `effb3` | `efficientnet-b3` | `train-unet-effb3-crops` | `unet_effb3_crops.pt` |
| `r34_v2` | `resnet34` | `train-unet-r34-v2` | `unet_r34_v2.pt` |
| `r34_v2_agree` | `resnet34` | `train-unet-r34-v2-agree` | `unet_r34_v2_agree.pt` |

It loads the checkpoint, predicts each holdout and test frame with overlapped tiles once per flip (4 network passes per frame), and writes the mean and the standard deviation of the flips to `/kaggle/working/probs/<MODEL>/`. After that, the post-processing sweeps, the bootstrap and the component selector run on CPU from these maps and never need the GPU for this model again.

The test maps are only stored. Nothing here scores or tunes on the test set: it has no labels, and the notebooks that use the cache choose settings on the holdout only. It does not train, does not write `submission.csv` and does not submit to Kaggle.

**Attach exactly these Kaggle inputs**

1. The competition `filament-segmentation-2026` (train images, train JSON and test images).
2. The dataset `ignacioaguirrecandia/solar-code` (the repo zip), a version that contains `filament.probcache`. An older version stops at the imports.
3. The output of the training notebook of `MODEL`: by default `train-unet-r34-crops-80ep`.

**Settings**

- Accelerator: **GPU T4**, a single T4.
- Internet: **On**, for the `pip install` of `segmentation-models-pytorch==0.6.0`. The checkpoint holds the encoder, so no encoder weights are downloaded.

**Outputs** (in `/kaggle/working/probs/<MODEL>/`)

- `holdout/` and `test/`, each with `mean.npy`, `std.npy` (`(N, 2048, 2048)` float16) and `files.json`.
- `manifest.json`: the weights file and hash, the inference settings, the seconds per image and the result of the cross-check.

For 142 holdout and 180 test [VERIFY] frames that is about 2.2 GiB and 2.8 GiB, and 7 files (8 if Kaggle extracts the checkpoint into a folder and the notebook has to re-zip it, which puts the re-zipped weights in the output too: leave that file out of the private dataset). The time per image is not measured yet [VERIFY]; the notebook prints it.

**Afterwards:** on the notebook's Output tab choose **New Dataset**, name it `solar-probs-<MODEL>` (with `_` kept in the model name replaced by `-` if Kaggle asks), and keep it **private**. The CPU notebooks attach that dataset. Do not commit these maps to git.

## 1. `src` on the path

`src` is the directory that contains `filament/crop_training.py`. A checkout in the working directory is used as it is. Otherwise the walk goes through `/kaggle/input` at any depth, skips every `competitions` directory, and takes the shallowest match. It is used in place and not copied.

In [ ]:
import os
import sys
from pathlib import Path


def _local_src() -> Path | None:
    """A checkout that already has the crop modules (a run off Kaggle)."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        src = candidate / "src"
        if (src / "filament" / "crop_training.py").is_file():
            return src
    return None


def _private_dataset_src(root: Path | None = None) -> Path | None:
    """Return the shallowest private-dataset ``src`` under ``root``.

    ``filament/crop_training.py`` may sit at any depth. Directories named
    ``competitions`` are not walked, so the MAGFiLO tree is left alone.
    """
    root = Path("/kaggle/input") if root is None else root
    if not root.is_dir():
        return None
    marker = Path("filament") / "crop_training.py"
    matches: list[Path] = []
    for dirpath, dirnames, _filenames in os.walk(root, topdown=True):
        dirnames[:] = [name for name in dirnames if name != "competitions"]
        current = Path(dirpath)
        if current.name != "src" or not (current / marker).is_file():
            continue
        matches.append(current)
        dirnames.clear()
    if not matches:
        return None
    return min(
        matches,
        key=lambda path: (len(path.relative_to(root).parts), path.as_posix()),
    )


def _put_on_path(src: Path) -> None:
    text = str(src)
    if text not in sys.path:
        sys.path.insert(0, text)
    current = os.environ.get("PYTHONPATH", "")
    parts = [part for part in current.split(os.pathsep) if part]
    if text not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([text, *parts]) if parts else text


def _prepare_src() -> Path:
    """Put the repo's ``src`` on ``sys.path`` and return it.

    A checkout in the working directory is used as it is. Otherwise the
    private dataset's ``src`` is used in place. It is not copied to
    ``/kaggle/working``: a copy there would be saved with this notebook's
    output and found again by a later notebook that attaches that output.
    """
    local = _local_src()
    if local is not None:
        _put_on_path(local)
        return local
    source = _private_dataset_src()
    if source is None:
        raise FileNotFoundError(
            "Could not find src/filament/crop_training.py in the working "
            "directory or under /kaggle/input (any depth outside "
            "competitions). Attach the dataset ignacioaguirrecandia/solar-code, "
            "for example "
            "/kaggle/input/datasets/ignacioaguirrecandia/solar-code/"
            "Solar-Filament-Segmentation-main/src. If it is attached, its "
            "version predates the crop modules: upload a new version of the "
            "dataset from the branch that has them. Do not make the GitHub "
            "repository public."
        )
    _put_on_path(source)
    return source


SRC = _prepare_src()
print(f"src={SRC}")
PYTHON = sys.executable

## 2. Install the segmentation library

`segmentation-models-pytorch` **0.6.0** (MIT), the version that trained the checkpoint. Kaggle already has PyTorch; this cell does not install it.

In [ ]:
import subprocess

# Pinned. segmentation-models-pytorch 0.6.0 is MIT.
SMP_VERSION = "0.6.0"

completed = subprocess.run(
    [
        PYTHON,
        "-m",
        "pip",
        "install",
        f"segmentation-models-pytorch=={SMP_VERSION}",
    ],
    check=False,
)
if completed.returncode != 0:
    raise RuntimeError(
        f"segmentation-models-pytorch=={SMP_VERSION} did not install. Stop. "
        "Internet has to be On. Kaggle already has PyTorch; this cell only "
        "installs segmentation-models-pytorch (MIT)."
    )

In [ ]:
import segmentation_models_pytorch as smp

print(f"segmentation-models-pytorch {smp.__version__}")
if smp.__version__ != SMP_VERSION:
    raise RuntimeError(
        f"Expected segmentation-models-pytorch {SMP_VERSION}, "
        f"found {smp.__version__}. Stop."
    )

## 3. Settings and where the weights are

- `MODEL` is a name of `filament.checkpoint.CROP_CHECKPOINTS`. `WEIGHTS_PATH` comes from that entry. The helpers that open a weights file, and re-zip a Kaggle-extracted archive, are the ones of `notebooks/infer_unet_crops.ipynb`, except that only a file with the name in that entry is accepted: a lone `.pt` of another run is never taken instead, since two runs with the same epochs and encoder pass the guard until the hash is pinned. The checkpoint is checked against its epochs, encoder and, once pinned, `tensor_sha256`; the notebook stops on a mismatch. A selector fitted on these maps is only valid for this exact checkpoint, so leave `ALLOW_UNVERIFIED_WEIGHTS` as `False`.
- `TILE_OVERLAP` (128), `INFER_BATCH_SIZE` (8) and `USE_AMP` are the values of the holdout evaluation in the training notebook. Tiles are averaged with equal weight, as there.
- `SPLITS` says what to export. `FULL_SIDE` is the frame size: a frame of another size stops the export.

In [ ]:
import zipfile

from filament.checkpoint import crop_checkpoint

# ---- Which model: a name of filament.checkpoint.CROP_CHECKPOINTS --------------
MODEL = "r34"
EXPECTED = crop_checkpoint(MODEL)

# ---- What to export ----------------------------------------------------------
SPLITS = ("holdout", "test")
FULL_SIDE = 2048

# ---- Weights and the checkpoint guard ---------------------------------------
# Default from the checkpoint config. Another .pt path loads that file; None
# searches /kaggle/working and /kaggle/input (skipping competitions).
WEIGHTS_PATH = EXPECTED.weights_path
ALLOW_UNVERIFIED_WEIGHTS = False

# ---- Inference ---------------------------------------------------------------
TILE_OVERLAP = 128
INFER_BATCH_SIZE = 8
USE_AMP = True
SEED = 0

OUTPUT_ROOT = Path("/kaggle/working/probs")
TEST_IMAGES = Path(
    "/kaggle/input/competitions/filament-segmentation-2026/"
    "MAGFiLO_1.0_Kaggle_2026/test/test_images"
)

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
PREFERRED_NAME = Path(EXPECTED.weights_path).name


def _pt_under(root: Path) -> list[Path]:
    """Every ``*.pt`` file or directory under ``root``.

    A directory counts because Kaggle extracts an uploaded ``.pt`` zip
    into a folder of that name. ``competitions`` is not walked.
    """
    found: list[Path] = []
    if not root.is_dir():
        return found
    for dirpath, dirnames, filenames in os.walk(root):
        directory = Path(dirpath)
        kept: list[str] = []
        for name in sorted(dirnames):
            if name == "competitions":
                continue
            if name.endswith(".pt"):
                found.append(directory / name)
                continue
            kept.append(name)
        dirnames[:] = kept
        for name in sorted(filenames):
            if name.endswith(".pt"):
                found.append(directory / name)
    return found


def _weight_candidates() -> list[Path]:
    """``.pt`` files and extracted archives, working then input."""
    return _pt_under(KAGGLE_WORKING) + _pt_under(KAGGLE_INPUT)


def _archive_error(path: Path, reason: str) -> RuntimeError:
    """Explain a Kaggle-extracted ``.pt`` that cannot be re-zipped."""
    return RuntimeError(
        f"{path} is a directory, not a checkpoint file. "
        "Kaggle extracts an uploaded .pt because torch.save writes a zip, "
        "so the file becomes a folder such as unet.pt/unet/data.pkl. "
        f"This notebook did not re-zip it: {reason} "
        "Set WEIGHTS_PATH to a real .pt file that Kaggle did not extract."
    )


def _rezip_extracted_archive(directory: Path, dest: Path | None = None) -> Path:
    """Re-zip one extracted torch archive to a file under working.

    Members keep the archive folder name (``unet/data.pkl``). The zip is
    stored, not deflated, so ``torch.load`` can read it. The directory
    itself is never returned.
    """
    if not directory.is_dir():
        raise _archive_error(directory, "the path is not a directory.")
    pkls = sorted(
        path
        for path in directory.rglob("data.pkl")
        if path.is_file() and not path.is_symlink()
    )
    if len(pkls) != 1:
        raise _archive_error(
            directory,
            f"expected one data.pkl, found {len(pkls)}.",
        )
    archive_dir = pkls[0].parent
    if archive_dir.parent == archive_dir:
        raise _archive_error(directory, "data.pkl has no parent folder.")
    names = {child.name for child in archive_dir.iterdir()}
    storage = (archive_dir / "data").is_dir()
    marked = "version" in names or "byteorder" in names or storage
    if "data.pkl" not in names or not marked:
        raise _archive_error(
            directory,
            "data.pkl is not next to version, byteorder, or a data folder.",
        )
    files: list[Path] = []
    ordered = sorted(archive_dir.rglob("*"), key=lambda item: item.as_posix())
    for path in ordered:
        if path.is_symlink():
            raise _archive_error(directory, f"{path.name} is a symlink.")
        if path.is_file():
            files.append(path)
    if not files:
        raise _archive_error(directory, "the archive folder has no files.")
    try:
        KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
    except OSError as exc:
        raise _archive_error(
            directory,
            f"/kaggle/working is not writable ({exc}).",
        ) from exc
    stem = directory.stem if directory.name.endswith(".pt") else directory.name
    if dest is None:
        dest = KAGGLE_WORKING / f"{stem}-from-extracted.pt"
    try:
        dest.resolve().relative_to(archive_dir.resolve())
    except ValueError:
        inside = False
    else:
        inside = True
    if inside:
        raise _archive_error(
            directory,
            "the re-zipped file would land inside the extracted archive.",
        )
    with zipfile.ZipFile(dest, "w", compression=zipfile.ZIP_STORED) as handle:
        for file in files:
            rel = file.relative_to(archive_dir.parent).as_posix()
            if rel.startswith("/") or ".." in rel.split("/"):
                raise _archive_error(directory, f"unsafe member {rel}.")
            handle.write(file, arcname=rel)
    print(f"Kaggle extracted the torch archive at {directory}. Re-zipped it to {dest}.")
    return dest


def _materialize_checkpoint(path: Path) -> Path:
    """Return a real ``.pt`` file. Re-zip an extracted archive."""
    if path.is_file():
        return path
    if path.is_dir():
        return _rezip_extracted_archive(path)
    raise FileNotFoundError(
        f"{path} does not exist. Set WEIGHTS_PATH to an existing .pt file."
    )


def _resolve_weights(explicit):
    """Return the checkpoint file to load.

    ``explicit`` that exists is used (an extracted archive is re-zipped). The
    default path missing, or ``None``, searches ``/kaggle/working`` and
    ``/kaggle/input`` for a file or folder named ``PREFERRED_NAME``. Another
    ``.pt`` is never taken instead: two runs of one recipe share the epochs and the
    encoder, so the guard cannot tell them apart until the hash is pinned. A path
    that was set by hand and is missing is an error.
    """
    if explicit is not None:
        path = Path(explicit)
        if path.exists():
            return _materialize_checkpoint(path)
        if str(explicit) != EXPECTED.weights_path:
            raise FileNotFoundError(
                f"WEIGHTS_PATH is set to {path}, but that file does not exist."
            )
        print(f"{path} does not exist. Searching for {PREFERRED_NAME}.")
    candidates = _weight_candidates()
    named = [path for path in candidates if path.name == PREFERRED_NAME]
    if len(named) == 1:
        return _materialize_checkpoint(named[0])
    listed = "\n".join(f"  {path}" for path in candidates) or "  (none)"
    detail = (
        f"Several files named {PREFERRED_NAME} were found."
        if named
        else f"No file named {PREFERRED_NAME} was found."
    )
    raise RuntimeError(
        f"{detail} Attach the output of the notebook "
        f"{Path(EXPECTED.weights_path).parent.name}, or "
        f"set WEIGHTS_PATH to the file to load.\n{listed}"
    )

## 4. Load the checkpoint and check it

Prints the stored fields and `tensor_sha256`, runs the guard, and rebuilds the network from the checkpoint with `encoder_weights=None`. The cell stops if there is no CUDA GPU, or if the attached code predates the cache.

In [ ]:
import random
import time

import numpy as np
import torch

from filament import crop_training as ct
from filament.checkpoint import checkpoint_fingerprint, verify_and_report
from filament.ensemble import FLIP_NAMES
from filament.probcache import (
    ProbCache,
    export_prob_maps,
    model_dir,
    split_dir,
    write_manifest,
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = ct.require_cuda(6.0)
print(f"device={torch.cuda.get_device_name(0)} torch={torch.__version__}")

weights_file = _resolve_weights(WEIGHTS_PATH)
print(f"loaded weights={weights_file}")
checkpoint = ct.load_checkpoint(weights_file)
fingerprint = checkpoint_fingerprint(checkpoint)
for key in ("epochs", "status", "encoder_name", "crop_size", "smp_version"):
    print(f"  {key}={checkpoint.get(key, 'not stored')}")
print(f"  tensor_sha256={fingerprint['tensor_sha256']}")
verify_and_report(
    fingerprint,
    allow_unverified=ALLOW_UNVERIFIED_WEIGHTS,
    expected=EXPECTED,
)
TILE_SIZE = int(checkpoint.get("tile_size") or checkpoint["crop_size"])
model = ct.model_from_checkpoint(checkpoint, device)
del checkpoint
predict = ct.tiled_predictor(
    model,
    device,
    tile_size=TILE_SIZE,
    overlap=TILE_OVERLAP,
    batch_size=INFER_BATCH_SIZE,
    use_amp=USE_AMP,
)
print(f"{MODEL}: tile={TILE_SIZE} overlap={TILE_OVERLAP} batch={INFER_BATCH_SIZE}")

## 5. Which frames

The holdout is the same split as the training notebook: the most recent 20% of calendar dates (142 files with 231 annotator views on the current data). The test frames are every JPEG in the competition's test folder.

In [ ]:
from filament.annotations import (
    TRAIN_ANNOTATIONS_PATH,
    TRAIN_IMAGES_DIR,
    load_train_annotations,
)
from filament.sweep import count_views
from filament.train_data import split_training_images

IMAGE_DIR = Path(TRAIN_IMAGES_DIR)
ANNOTATIONS_PATH = Path(TRAIN_ANNOTATIONS_PATH)
if not ANNOTATIONS_PATH.is_file() or not IMAGE_DIR.is_dir():
    raise FileNotFoundError(
        "Train data was not found. Checked "
        f"images={IMAGE_DIR} and annotations={ANNOTATIONS_PATH}. "
        "Attach the competition filament-segmentation-2026 to this notebook."
    )
if not TEST_IMAGES.is_dir():
    raise FileNotFoundError(
        f"Test images were not found at {TEST_IMAGES}. Attach the competition."
    )
loaded = load_train_annotations(ANNOTATIONS_PATH)
split = split_training_images(loaded.images)
holdout_views = count_views(split.holdout)
print(
    f"train files={len(split.train)} holdout files={len(split.holdout)} "
    f"holdout annotator views={holdout_views}"
)
if (len(split.train), len(split.holdout), holdout_views) != (565, 142, 231):
    print(
        "WARNING: the split is not 565 / 142 files with 231 views, the numbers "
        "the training notebook used. [VERIFY] the data version."
    )
SPLIT_PATHS = {
    "holdout": [IMAGE_DIR / image.file_name for image in split.holdout],
    "test": sorted(
        path
        for path in TEST_IMAGES.iterdir()
        if path.is_file() and path.suffix.lower() == ".jpeg"
    ),
}
for name in SPLITS:
    print(f"{name}: {len(SPLIT_PATHS[name])} frames")
    if not SPLIT_PATHS[name]:
        raise FileNotFoundError(f"No frames for the split {name!r}.")

## 6. Predict and write the maps

Each frame goes through the network once per flip (`none`, `h`, `v`, `hv`), and the prediction is flipped back. `mean.npy` holds the mean of the four maps, which is the map flip TTA scores, and `std.npy` their standard deviation. A progress line is printed every 20 frames.

In [ ]:
def _progress(index, total, path):
    if index == 1 or index % 20 == 0 or index == total:
        print(f"  {index}/{total} {path.name}", flush=True)


timings = {}
for name in SPLITS:
    print(f"exporting {name}")
    timings[name] = export_prob_maps(
        SPLIT_PATHS[name],
        predict,
        split_dir(OUTPUT_ROOT, MODEL, name),
        shape=(FULL_SIDE, FULL_SIDE),
        progress=_progress,
    )
    print(
        f"{name}: {timings[name].files} frames, "
        f"{timings[name].seconds_per_image:.2f} s per image for inference, "
        f"{timings[name].total_seconds:.0f} s in total"
    )

## 7. The manifest

Written as soon as the maps are finished, before the cross-check, so a failure in the next cell cannot leave a cache without one. It records which file the weights came from, their hash, and the inference settings. A selector fitted on these maps is only valid for exactly this checkpoint.

In [ ]:
MANIFEST = {
    "model": MODEL,
    "label": EXPECTED.label,
    "encoder": fingerprint["encoder"],
    "epochs": fingerprint["epochs"],
    "tensor_sha256": fingerprint["tensor_sha256"],
    "weights_file": str(weights_file),
    "tile_size": TILE_SIZE,
    "tile_overlap": TILE_OVERLAP,
    "tile_window": "uniform",
    "infer_batch_size": INFER_BATCH_SIZE,
    "use_amp": USE_AMP,
    "flips": list(FLIP_NAMES),
    "dtype": "float16",
    "smp_version": smp.__version__,
    "torch_version": torch.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "cross_check": "not run",
    "splits": {
        name: {
            "files": timing.files,
            "inference_seconds": round(timing.inference_seconds, 1),
            "seconds_per_image": round(timing.seconds_per_image, 3),
        }
        for name, timing in timings.items()
    },
}
manifest_path = write_manifest(OUTPUT_ROOT, MODEL, MANIFEST)
print(f"manifest={manifest_path}")

## 8. Cross-check on the holdout

The cached holdout maps go through the existing scoring (`filament.sweep.sweep_postproc_grid`, pooled per-view PQ), which reads them back from disk. For `r34`, threshold 0.45, `close_size` 11, `min_area` 400 should give about 0.409, the flip-TTA figure in `docs/PROJECT_REPORT.md` (row 12, from Nacho's notes; its CSV is not in the repo [VERIFY]). That figure has no tolerance here. What the notebook does check is a floor: at threshold 0.5, `close_size` 0, `min_area` 400, flip TTA should not score much below the checkpoint's recorded `holdout_pq`, which is without TTA. If it does, the maps do not line up with their frames, or the weights are not the ones of that run: the manifest then says `cross_check: FAILED`, and none of the later notebooks should be trusted with this cache. Storing the maps as float16 can move a pixel only when it lies right next to the threshold.

Only the holdout is scored here. Nothing reads a score from the test set.

In [ ]:
from filament.crop_data import read_gray
from filament.sweep import sweep_postproc_grid

# TTA only adds to the no-TTA PQ of the training notebook, so a drop of more than
# this means something is wrong, not noise.
CROSS_CHECK_FLOOR_MARGIN = 0.03

if "holdout" in SPLITS:
    cache = ProbCache(split_dir(OUTPUT_ROOT, MODEL, "holdout"))
    frames = iter(split.holdout)

    def _read(image):
        return read_gray(IMAGE_DIR / image.file_name, (image.height, image.width))

    def _cached(_gray):
        return cache.mean(next(frames).file_name)

    begin = time.monotonic()
    totals = sweep_postproc_grid(
        split.holdout, _read, _cached, (0.45, 0.5), (0, 11), (400,)
    )
    scores = {}
    for point, counts in totals.items():
        scores[(point.threshold, point.close_size, point.min_area)] = counts.pq
        print(
            f"holdout, flip TTA from the cache: threshold={point.threshold} "
            f"close_size={point.close_size} min_area={point.min_area} "
            f"pooled_pq={counts.pq:.4f}"
        )
    print(f"cross-check took {time.monotonic() - begin:.0f} s")
    MANIFEST["cross_check_pq"] = {
        f"threshold={t:g} close_size={k} min_area={a}": round(float(v), 4)
        for (t, k, a), v in scores.items()
    }
    floor_pq = scores[(0.5, 0, 400)]
    if EXPECTED.holdout_pq is None:
        MANIFEST["cross_check"] = "no recorded holdout_pq to compare with"
    elif floor_pq >= EXPECTED.holdout_pq - CROSS_CHECK_FLOOR_MARGIN:
        MANIFEST["cross_check"] = "ok"
    else:
        MANIFEST["cross_check"] = "FAILED"
        print(
            f"WARNING: cross-check FAILED. PQ {floor_pq:.4f} at (0.5, 0, 400) is more "
            f"than {CROSS_CHECK_FLOOR_MARGIN} below the recorded {EXPECTED.holdout_pq}. "
            "Do not use this cache until that is explained."
        )
    print(f"cross-check: {MANIFEST['cross_check']}")
else:
    MANIFEST["cross_check"] = "holdout not exported"

write_manifest(OUTPUT_ROOT, MODEL, MANIFEST)
print(manifest_path.read_text())
size = sum(
    path.stat().st_size
    for path in model_dir(OUTPUT_ROOT, MODEL).rglob("*")
    if path.is_file()
)
print(f"output size={size / 2**30:.2f} GiB")
print("This notebook wrote no submission file and did not submit to Kaggle.")